# onceover

Onceover helps you understand an unfamiliar table before you join it,
clean it, or build a model. One call profiles a Polars DataFrame, a LazyFrame, or a
file (Parquet, CSV, or IPC) and returns column summaries, ranked
findings, and the evidence behind them.
The computation runs in Rust; the same profiler is available from Python
and Rust.

A useful first pass can answer more than “which cells are null?” It can
find a key that almost works, a column derived from other columns, a rule
that holds within a segment, an unusual observation, or a change in a
relationship over time. The report keeps exact comparisons, predictive
estimates, and suggested interpretations separate.


## Version

This book is built by executing its examples against the installed
package, and its detector registry is generated from that same
build. The version that produced this build is:

In [1]:
import onceover

onceover.__version__

'0.1.6'

In [2]:
import polars as pl
import onceover

n = 600
quantity = [1 + i % 20 for i in range(n)]
region = [i % 3 for i in range(n)]
df = pl.DataFrame({
    "order_id": range(n),
    "quantity": quantity,
    "double_quantity": [2 * q for q in quantity],
    "region_code": region,
    "region_name": [["north", "south", "west"][r] for r in region],
})
report = onceover.profile(df)
print(report.to_digest(1_600))


profile digest: 600 rows x 5 columns, 39 findings, schema 0.8
What a row is:
order_id uniquely identifies rows
What is redundant:
double_quantity = 2·quantity within tolerance on all 600 non-missing row pairs [dependencies-b097b740f5504a9d]
region_code and region_name encode the same grouping: 3 levels, paired one-to-one on all 600 rows [column_roles-9bdbcb8e4c889830]
Other relationships:
order_id uniquely identifies rows (primary key candidate) [uniqueness-1ac6146d2c27c2e2]
numeric column region_code behaves like a category (3 distinct values) [column_roles-49ad4d241e8d8fdc]



## Find the answer you need

**Check the table's integrity.** Find candidate and nearly unique keys,
repeated rows, missing-value codes, inconsistent formats, rounding changes,
and unusually isolated values. Start with
[keys and duplicates](methodology/keys_duplicates.ipynb),
[column cards](usage/columns.ipynb), and
[extreme values](methodology/extreme_values.ipynb).

**Understand relationships.** Discover lookup mappings, arithmetic
identities, predictable targets, conditional rules, concentrated
combinations, local equations, predictor interactions, and shared row
states. The [capability guide](usage/capabilities.ipynb) connects those
questions to the individual methods.

**Explain changes.** Compare associations across periods, locate changes
in a time-ordered series, detect threshold jumps, and inspect drift in the
inputs used by a reference-period model. Begin with
[temporal analysis](methodology/temporal_analysis.ipynb),
[period associations](methodology/period_associations.ipynb), and
[model-routing drift](methodology/drift.ipynb).

**Check intended structure.** Declare a calendar, ordering, complete
sequence, or panel layout with `hints`. The
[expectations pass](methodology/expectations.ipynb) checks what the caller
knows about the table's intended structure.

## From a first look to the method

1. Follow the [quickstart](usage/quickstart.ipynb) to profile a table and
   inspect a complete finding.
2. Use the [capability guide](usage/capabilities.ipynb) to choose a question
   and find the relevant detector.
3. Read [findings](usage/findings.ipynb) and
   [coverage](usage/coverage.ipynb) together: a missing finding may reflect
   a sample, a search cap, a deadline stop, or a pass that did not run.
4. Open the linked methodology chapter for the calculation, its evidence,
   the assumptions it makes, and a runnable example.

The [case studies](case_studies/vendor_extract.ipynb) show how findings
inform decisions about vendor extracts, redundant features, dated shifts,
and an expanding delivery service. The
[implementation chapters](implementation/architecture.ipynb) explain the
Rust pipeline. The [API reference](usage/api.ipynb),
[JSON contract](appendix/report_contract.ipynb), and
[glossary](appendix/glossary.ipynb) provide lookup details.

## Read the report at the right level

`report.to_digest()` gives a short index. `report.select(...)` retrieves
complete findings. `report.columns` describes every column, and
`report.coverage` explains the work behind the result. For each finding,
check the evaluated rows and validation before applying it to new data.
A high rank is reporting priority, not a probability of correctness.

The [analysis notebook](https://github.com/fcela/onceover/blob/main/notebooks/onceover_analysis.ipynb)
provides a longer worked example with charts. The
[large-table guide](usage/large_tables.ipynb) explains how
tables with millions of rows are profiled from a bounded sample whose
size and seed are recorded, including relationships beyond column 64.
